In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from random import randint

from rich.pretty import pprint

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import fdl_classes as fdc

## Load dataframe

In [ ]:
csv = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv"))
csv.head()

## Check label columns

In [ ]:
pprint(csv.columns[1:])

## Test Dataset

In [ ]:
path_to_images = Path(".").joinpath("data").joinpath("images")
path_to_images.is_dir()

In [ ]:
dataset = fdc.FldDataset(dataframe = csv, train_mode=True,test_mode=True)

In [ ]:
pprint(dataset.transform)

In [ ]:
plt.imshow(dataset[randint(0, len(dataset) - 1)]["image"])

## Train

### Split dataset

In [ ]:
train, val = train_test_split(csv, test_size=0.3)
val, test = train_test_split(val, test_size=0.5)

for d in [train, val, test]:
    print(d.shape)

### Create model

In [ ]:
model = fdc.FdlNet(
    batch_size=10,
    learning_rate=0.001,
    max_epochs=2,
    num_workers=10,
    train=train,
    val=val,
    test=test,
)

### Create Trainer

In [ ]:
trainer = fdc.get_trainer(model=model, checkpoints_path=Path("."))

### Train

In [ ]:
model.encoder = model.encoder.train()
trainer.fit(model)

In [ ]:
pprint(trainer.checkpoint_callback.best_model_path)
model = fdc.FdlNet.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path, weights_only=False
)

### Evaluate

In [ ]:
data = model.classification_report_as_dict()

pd_data = {"labels": [], "precision": [], "recall": [], "f1-score": [], "support": []}
for k, v in data.items():
    pd_data["labels"].append(k)
    for c in ["precision", "recall", "f1-score", "support"]:
        pd_data[c].append(v[c])

pd.DataFrame(data=pd_data)